# TMDWF Joint CS-Kernel Template

This notebook is a standalone wrapper around the repository's joint downstream TMDWF CS-kernel workflow.
It reads already-generated Fourier bootstrap samples from one or more ensembles and fits `gamma_MSbar(x, bT)` independently at each specified x, parameterizing the bT-dependence with a 1D spline without first running the per-ensemble CS-kernel step.

## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_cs_kernel_joint_input_text,
    run_tmdwf_cs_kernel_joint_from_notebook,
    validate_tmdwf_cs_kernel_joint_notebook_config,
)


## User Inputs

These fields describe a joint CS-kernel fit.
The workflow expects repository-native TMDWF Fourier outputs to already exist under each ensemble's `input_root`.


In [ ]:
workflow_config = {
    # Shared Fourier-output settings
    "gm": "T5",
    "eta": "eta0",
    "component": "real",
    "nstates": 2,
    "normalization_mode": "mode3",

    # Joint CS-kernel settings
    "mu": 2.0,
    "scheme": "CG",
    "kernel_label": "LO",
    "reference_p1_gev": 1.0,
    "x_window": [0.2, 0.8],
    "x_knots": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "bT_knots_fm": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    "spline_kind": "linear",
    "plot": True,
    "progress": True,
    "progress_every": 10,

    # Multiplicative systematic corrections on the matrix element
    "fit_a2_correction": True,
    # "fit_fv_correction": True,
    "fit_pz2_correction": True,
    # "fit_apz2_correction": True,
    "a2_correction_prior_width": 1.0,
    "fv_correction_prior_width": 1.0,
    "pz2_correction_prior_width": 1.0,
    "apz2_correction_prior_width": 1.0,

    # One entry per ensemble
    "ensembles": [
        {
            "label": "l48a060",
            "input_root": str(REPO_ROOT / "analysis_l48c64a060_m140_src5" / "4-FT-new"),
            "title_pattern": "l48c64a060_m140_fit_pz*",
            "ns": 48,
            "lattice_spacing_fm": 0.060,
            "pzrange": [1, 5],
            "bTrange": [1, 20],
            "m_pi_mev": 140.0,
        },
        {
            "label": "l64a050",
            "input_root": str(REPO_ROOT / "analysis_l64c64a050_m140_src5" / "4-FT-new"),
            "title_pattern": "l64c64a050_m140_fit_pz*",
            "ns": 64,
            "lattice_spacing_fm": 0.050,
            "pzrange": [3, 8],
            "bTrange": [1, 30],
            "m_pi_mev": 140.0,
        },
    ],

    # Output settings
    "results_dir": str(REPO_ROOT / "results_tmdwf_cs_kernel_joint"),
}
workflow_config

## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `gm`, `eta`, `component`, `nstates`, `normalization_mode`: Select the Fourier output family to consume. These settings are shared by all ensembles in the joint fit.
- `mu`, `scheme`, `kernel_label`: Matching settings used in the type-2 correction. The first version expects one `kernel_label` per run.
- `reference_p1_gev`: Physical reference momentum scale in GeV used in the direct evolution formula. This is not a lattice momentum integer.
- `x_window`: Observations whose actual data-grid x value falls outside this window are excluded.
- `x_knots`: The x values at which `gamma_MSbar` is fitted independently. Each is mapped to the nearest point on the data x-grid; the actual x value is recorded in the output. If omitted, the workflow picks up to 6 evenly spaced points within `x_window`.
- `bT_knots_fm`: Spline knots (in fm) for the bT-direction spline parameterizing `gamma_MSbar(bT)` at each x. If omitted, the workflow uses the unique physical bT values across all ensembles, capped at 8 knots.
- `spline_kind`: Interpolation kind for the gamma bT spline. Use `linear` for the piecewise-linear hat basis or `cubic` for a natural cubic spline basis.
- `fit_a2_correction`, `fit_pz2_correction`: Central multiplicative correction model. Each enabled channel adds two dimensionless analytic parameters, not a bT spline. The correction factor multiplies `gamma_MSbar(bT)` inside the exponent: `exp(log(pz/P1) * (gamma * C_corr - delta_matching))`. The fixed reference scales are `a0=0.1 fm`, `b0=1.0 fm`, and `p0=1.0 GeV`; the active shapes in `C_corr` are `(a/a0)^2 * [c0 + c1 * (b0/bT)^2]` and `(1/x^2 + 1/(1-x)^2) * (p0/pz)^2 * [c0 + c1 * (b0/bT)^2]`.
- `fit_fv_correction`, `fit_apz2_correction`: Optional systematic variations. The finite-volume factor is `exp(-M_pi L) / sqrt(M_pi L)` times `beta0 + beta1 exp(M_pi bT)`, with `M_pi bT` in the same units as `M_pi L`. The a2 and pz2 corrections use inverse bT, so selected fit data must exclude `bT=0` when either is enabled. The apz2 variation uses one lambda coefficient.
- `a2_correction_prior_width`, `fv_correction_prior_width`, `pz2_correction_prior_width`, `apz2_correction_prior_width`: Zero-centered Gaussian prior widths for the enabled correction parameters. Each prior is applied only to its own channel's nuisance coefficients and does not modify the main `gamma_MSbar` spline.
- `plot`: Whether to write one `gamma_MSbar` vs x band plot for each `bT_knots_fm` value, plus per-x diagnostic plots showing data vs fit in pz-space for each `(ensemble, bT)` group with at least three momentum values.
- `progress`, `progress_every`: Print bootstrap-fit progress (per x and per sample) while running. If `progress_every` is omitted, the workflow reports roughly every 5% of the bootstrap samples.
- `ensembles`: One dictionary per ensemble. Each dictionary gives the ensemble label, Fourier output root, per-pz title pattern, `Ns`, lattice spacing, and either `pzlist`/`bTlist` or inclusive `pzrange`/`bTrange`.
- `results_dir`: Output root for the joint summary, surface table, bootstrap surface samples, spline coefficients, and diagnostics.

Expected input data shape:

- The workflow reads repository-native Fourier sample tables with one row per `(sample_id, x)` and a `q_sample` column.
- Each ensemble keeps its own physical `Pz` and physical `bT = nT * a`; the fit does not force ensembles onto a common lattice grid.
- At each x, one nuisance amplitude is eliminated analytically for each `(ensemble, bT)` group while the bT-spline coefficients are fit via nonlinear least squares.

What the workflow writes:

- `joint_gamma_eff/*_summary.txt`
- `joint_gamma_eff/tables/*_surface.txt`
- `joint_gamma_eff/samples/*_samples.txt`
- `joint_gamma_eff/samples/*_coefficients.txt`
- `joint_gamma_eff/diagnostics/*_diagnostics.txt`
- `joint_gamma_eff/plots/*_x_band.pdf`
- `joint_gamma_eff/plots/diagnostics/*_x{X}_*_pz_diagnostics.pdf`

The `*_coefficients.txt` file records the bT-spline coefficients for every bootstrap sample at each x. Together with `bT_knots_fm` and `spline_kind` from `*_summary.txt`, they can reconstruct `gamma_MSbar(bT)` at arbitrary bT values. Enabled correction coefficient files record the analytic coefficients for each correction channel.

The `*_pz_diagnostics.pdf` files show, for each fitted x, the data points and reconstructed fit band in `O vs pz` space. Each panel corresponds to one `(ensemble, bT)` group with at least three momentum values, allowing visual inspection of fit quality across the momentum range.

## Validate Config

This uses the same parser as the CLI workflow, so it is a good way to confirm the text rendering and defaults before running.


In [ ]:
validated = validate_tmdwf_cs_kernel_joint_notebook_config(workflow_config)
validated


## Render Input Preview

This is the plain-text control file that the notebook helper materializes behind the scenes.


In [ ]:
input_preview = render_tmdwf_cs_kernel_joint_input_text(workflow_config)
print(input_preview)


## Run Workflow

This launches the repository-native joint CS-kernel workflow and prints the generated artifacts.


In [ ]:
# outputs = run_tmdwf_cs_kernel_joint_from_notebook(workflow_config)
# for output in outputs:
#     print(output)


## Config Snapshot

This is useful to keep alongside saved results.


In [ ]:
print(pretty_print_config(workflow_config))
